# EDA multivariable por estaciones — Cali 2022

Este notebook replica la estructura del EDA original por estaciones, pero queda parametrizado para trabajar con varias variables de calidad del aire y meteorológicas en un solo año.

La lógica general es:

1. Cargar automáticamente las bases configuradas.
2. Estandarizar estación, fecha y valor.
3. Filtrar el año de estudio.
4. Calcular calidad temporal, duplicados, nulos, negativos y faltantes por estación.
5. Generar tablas, gráficos, huecos temporales, ACF/PACF, ADF y descomposición STL.
6. Construir una base integrada multivariable usando las estaciones comunes.

> **Uso:** sube las bases CSV/Excel al mismo directorio del notebook y ejecuta todas las celdas. Si un archivo cambia de nombre, solo modifica el diccionario `ARCHIVOS`.

## 1. Cargar librerías

In [ ]:
import os
import re
import csv
import warnings
from pathlib import Path
from itertools import groupby

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import STL

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["axes.grid"] = True


## 2. Parámetros del análisis

In [ ]:
CIUDAD = "Cali"
ANIO = 2022
FRECUENCIA_ESPERADA = "D"  # "D" = diaria. Cambiar a "H" si las bases vienen horarias.
UMBRAL_ROJO = 25           # porcentaje de faltantes para clasificar una estación como roja.
HUECO_CORTO_MAX_DIAS = 4   # criterio usado para identificar estaciones con huecos cortos.

# Carpeta donde están los archivos. En Colab normalmente es /content; en Jupyter puede ser la misma carpeta del notebook.
CARPETA_DATOS = Path(".")
CARPETA_SALIDAS = Path(f"salidas_eda_{CIUDAD.lower().replace(' ', '_')}_{ANIO}")
CARPETA_SALIDAS.mkdir(exist_ok=True)

# Configuración de archivos.
# variable_final: nombre limpio que tendrá la columna en la base integrada.
# archivo: nombre del archivo cargado.
# columna_valor: nombre de la columna original donde está la medición.
ARCHIVOS = {'DViento': {'archivo': 'DViento(11).csv', 'columna_valor': 'DViento'},
 'HAire10': {'archivo': 'HAire10(4).csv', 'columna_valor': 'HAire10'},
 'PLiquida': {'archivo': 'PLiquida(9).csv', 'columna_valor': 'P'},
 'PM10': {'archivo': 'PM10(12).csv', 'columna_valor': 'PM10'},
 'PM25': {'archivo': 'PM25(5).csv', 'columna_valor': 'PM2.5'},
 'RGlobal': {'archivo': 'RGlobal(6).csv', 'columna_valor': 'RGlobal'},
 'TMPR_AIR_10CM': {'archivo': 'Tem_Air_10cm.csv', 'columna_valor': 'TMPR AIR 10CM'},
 'VViento': {'archivo': 'VViento(14).csv', 'columna_valor': 'VViento'}}

ARCHIVOS


### Interpretación
En este bloque se definen la ciudad, el año, la frecuencia temporal esperada y los nombres de las bases. Para replicar el EDA con otra ciudad o año, solo se modifica esta sección.

## 3. Funciones auxiliares de carga, limpieza y diagnóstico

In [ ]:
def normalizar_texto(x):
    """Limpia textos de estaciones y columnas sin perder tildes."""
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    x = re.sub(r"\s+", " ", x)
    return x.upper()


def detectar_separador_y_encoding(ruta):
    raw = Path(ruta).read_bytes()[:10000]
    encoding = "utf-8-sig"
    for enc in ["utf-8-sig", "utf-8", "latin1"]:
        try:
            raw.decode(enc)
            encoding = enc
            break
        except Exception:
            continue
    texto = raw.decode(encoding, errors="ignore")
    try:
        dialect = csv.Sniffer().sniff(texto, delimiters=",;\t|")
        sep = dialect.delimiter
    except Exception:
        sep = ","
    return sep, encoding


def leer_base(ruta):
    ruta = Path(ruta)
    if not ruta.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {ruta}")
    if ruta.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(ruta)
    sep, encoding = detectar_separador_y_encoding(ruta)
    return pd.read_csv(ruta, sep=sep, encoding=encoding)


def buscar_columna(df, posibles):
    cols = list(df.columns)
    mapa = {str(c).strip().lower(): c for c in cols}
    for posible in posibles:
        key = str(posible).strip().lower()
        if key in mapa:
            return mapa[key]
    # búsqueda flexible
    for c in cols:
        c_norm = str(c).strip().lower()
        for posible in posibles:
            p_norm = str(posible).strip().lower()
            if p_norm in c_norm:
                return c
    return None


def convertir_numero(serie):
    if pd.api.types.is_numeric_dtype(serie):
        return pd.to_numeric(serie, errors="coerce")
    s = serie.astype(str).str.strip()
    s = s.str.replace("\u00a0", "", regex=False)
    # Si viene con coma decimal, se transforma a punto decimal.
    s = s.str.replace(",", ".", regex=False)
    return pd.to_numeric(s, errors="coerce")


def preparar_base(variable_final, cfg):
    ruta = CARPETA_DATOS / cfg["archivo"]
    df_raw = leer_base(ruta)
    df_raw.columns = [str(c).strip() for c in df_raw.columns]

    col_estacion = buscar_columna(df_raw, ["Estacion", "Estación", "station", "nombre estacion", "nombre estación"])
    col_fecha = buscar_columna(df_raw, ["Fecha inicial", "Fecha", "fecha", "date", "datetime"])
    col_valor = cfg.get("columna_valor") or buscar_columna(df_raw, [variable_final])

    if col_estacion is None:
        raise ValueError(f"No se encontró columna de estación en {cfg['archivo']}. Columnas: {df_raw.columns.tolist()}")
    if col_fecha is None:
        raise ValueError(f"No se encontró columna de fecha en {cfg['archivo']}. Columnas: {df_raw.columns.tolist()}")
    if col_valor not in df_raw.columns:
        raise ValueError(f"No se encontró la columna de valor '{col_valor}' en {cfg['archivo']}. Columnas: {df_raw.columns.tolist()}")

    df = pd.DataFrame({
        "Ciudad": CIUDAD,
        "Variable": variable_final,
        "Estacion": df_raw[col_estacion].map(normalizar_texto),
        "Fecha": pd.to_datetime(df_raw[col_fecha], errors="coerce"),
        "Valor": convertir_numero(df_raw[col_valor]),
    })

    df = df.dropna(subset=["Estacion", "Fecha"])
    df["Año"] = df["Fecha"].dt.year
    df = df[df["Año"] == ANIO].copy()
    df = df.sort_values(["Estacion", "Fecha"])

    duplicados = df.duplicated(subset=["Estacion", "Fecha"]).sum()
    if duplicados > 0:
        print(f"{variable_final}: se encontraron {duplicados} duplicados por Estacion + Fecha. Se agregan con promedio diario.")
        df = (
            df.groupby(["Ciudad", "Variable", "Estacion", "Fecha", "Año"], as_index=False)
              .agg(Valor=("Valor", "mean"))
        )

    return df


def calendario_esperado(anio=ANIO, frecuencia=FRECUENCIA_ESPERADA):
    inicio = pd.Timestamp(f"{anio}-01-01")
    fin = pd.Timestamp(f"{anio}-12-31")
    return pd.date_range(inicio, fin, freq=frecuencia)


def clasificar_estacion(pct_faltantes):
    if pct_faltantes == 0:
        return "Completa"
    if pct_faltantes >= UMBRAL_ROJO:
        return "Roja"
    return "Amarilla"


def estilo_estado(row):
    estado = row.get("Estado", "")
    if estado == "Roja":
        return ["background-color: #f4cccc"] * len(row)
    if estado == "Amarilla":
        return ["background-color: #fff2cc"] * len(row)
    if estado == "Completa":
        return ["background-color: #d9ead3"] * len(row)
    return [""] * len(row)


def resumen_cobertura(df_var):
    fechas = calendario_esperado()
    esperadas = set(fechas)
    filas = []
    for estacion, g in df_var.groupby("Estacion"):
        obs = set(pd.DatetimeIndex(g["Fecha"].dropna().unique()))
        faltantes = sorted(esperadas - obs)
        pct = len(faltantes) / len(fechas) * 100 if len(fechas) else np.nan
        filas.append({
            "Variable": g["Variable"].iloc[0],
            "Estacion": estacion,
            "Fecha_min": g["Fecha"].min(),
            "Fecha_max": g["Fecha"].max(),
            "Registros_observados": len(g),
            "Dias_con_dato": g["Fecha"].nunique(),
            "Dias_esperados": len(fechas),
            "Dias_faltantes": len(faltantes),
            "%_faltantes": round(pct, 2),
            "Nulos_variable": int(g["Valor"].isna().sum()),
            "Valores_negativos": int((g["Valor"] < 0).sum()),
            "Estado": clasificar_estacion(round(pct, 10)),
        })
    return pd.DataFrame(filas).sort_values(["Variable", "%_faltantes", "Estacion"])


def resumen_estadistico(df_var):
    return (
        df_var.groupby(["Variable", "Estacion"])
        .agg(
            fecha_min=("Fecha", "min"),
            fecha_max=("Fecha", "max"),
            registros=("Valor", "size"),
            datos_validos=("Valor", "count"),
            media=("Valor", "mean"),
            mediana=("Valor", "median"),
            desviacion=("Valor", "std"),
            minimo=("Valor", "min"),
            q1=("Valor", lambda x: x.quantile(0.25)),
            q3=("Valor", lambda x: x.quantile(0.75)),
            maximo=("Valor", "max"),
        )
        .reset_index()
        .round(3)
    )


def agrupar_fechas_consecutivas(fechas):
    if len(fechas) == 0:
        return []
    fechas = pd.to_datetime(sorted(fechas))
    grupos = []
    inicio = fechas[0]
    anterior = fechas[0]
    for fecha in fechas[1:]:
        if (fecha - anterior).days == 1:
            anterior = fecha
        else:
            grupos.append((inicio, anterior, (anterior - inicio).days + 1))
            inicio = fecha
            anterior = fecha
    grupos.append((inicio, anterior, (anterior - inicio).days + 1))
    return grupos


def tabla_huecos(df_var):
    fechas = calendario_esperado()
    esperadas = set(fechas)
    resumen = []
    detalle = []
    for estacion, g in df_var.groupby("Estacion"):
        obs = set(pd.DatetimeIndex(g["Fecha"].dropna().unique()))
        faltantes = sorted(esperadas - obs)
        bloques = agrupar_fechas_consecutivas(faltantes)
        if bloques:
            for i, (ini, fin, dias) in enumerate(bloques, start=1):
                detalle.append({
                    "Variable": g["Variable"].iloc[0],
                    "Estacion": estacion,
                    "Bloque": i,
                    "Inicio_hueco": ini,
                    "Fin_hueco": fin,
                    "Duracion_dias": dias,
                })
        resumen.append({
            "Variable": g["Variable"].iloc[0],
            "Estacion": estacion,
            "Cantidad_bloques_huecos": len(bloques),
            "Total_dias_faltantes": len(faltantes),
            "Hueco_mas_largo_dias": max([b[2] for b in bloques], default=0),
        })
    return pd.DataFrame(resumen), pd.DataFrame(detalle)


def serie_diaria_estacion(df_var, estacion):
    fechas = calendario_esperado()
    g = df_var[df_var["Estacion"] == estacion].copy()
    s = g.set_index("Fecha")["Valor"].sort_index()
    s = s.reindex(fechas)
    s.index.name = "Fecha"
    return s


## 4. Cargar todas las bases configuradas

In [ ]:
bases = {}
inventario = []

for variable_final, cfg in ARCHIVOS.items():
    try:
        df_var = preparar_base(variable_final, cfg)
        bases[variable_final] = df_var
        inventario.append({
            "Variable": variable_final,
            "Archivo": cfg["archivo"],
            "Columna_valor_original": cfg.get("columna_valor"),
            "Registros": len(df_var),
            "Estaciones": df_var["Estacion"].nunique() if len(df_var) else 0,
            "Fecha_min": df_var["Fecha"].min() if len(df_var) else pd.NaT,
            "Fecha_max": df_var["Fecha"].max() if len(df_var) else pd.NaT,
        })
    except Exception as e:
        inventario.append({
            "Variable": variable_final,
            "Archivo": cfg.get("archivo"),
            "Error": str(e)
        })

inventario_df = pd.DataFrame(inventario)
display(inventario_df)
inventario_df.to_csv(CARPETA_SALIDAS / "00_inventario_archivos.csv", index=False, encoding="utf-8-sig")

if not bases:
    raise RuntimeError("No se cargó ninguna base. Revisa los nombres de archivos y columnas en ARCHIVOS.")

base_larga = pd.concat(bases.values(), ignore_index=True)
print("Dimensión de la base larga:", base_larga.shape)
display(base_larga.head())


### Interpretación
Esta tabla confirma qué archivos se cargaron, cuántos registros contiene cada variable, el rango temporal disponible y cuántas estaciones aparecen por variable.

## 5. Vista general por variable

In [ ]:
vista_general = (
    base_larga.groupby("Variable")
    .agg(
        registros=("Valor", "size"),
        datos_validos=("Valor", "count"),
        estaciones=("Estacion", "nunique"),
        fecha_min=("Fecha", "min"),
        fecha_max=("Fecha", "max"),
        valor_min=("Valor", "min"),
        valor_max=("Valor", "max"),
        media=("Valor", "mean"),
        nulos=("Valor", lambda x: x.isna().sum()),
        negativos=("Valor", lambda x: (x < 0).sum()),
    )
    .reset_index()
    .round(3)
)

display(vista_general)
vista_general.to_csv(CARPETA_SALIDAS / "01_vista_general_variables.csv", index=False, encoding="utf-8-sig")


### Interpretación
La vista general permite revisar rápidamente si las bases corresponden al año esperado, si existen valores nulos o negativos y qué variables tienen una sola estación o varias estaciones disponibles.

## 6. Calidad temporal, faltantes y cobertura por estación

In [ ]:
tablas_cobertura = []
tablas_estadisticas = []
tablas_huecos_resumen = []
tablas_huecos_detalle = []

for variable, df_var in bases.items():
    cobertura = resumen_cobertura(df_var)
    estadistica = resumen_estadistico(df_var)
    huecos_resumen, huecos_detalle = tabla_huecos(df_var)

    tablas_cobertura.append(cobertura)
    tablas_estadisticas.append(estadistica)
    tablas_huecos_resumen.append(huecos_resumen)
    if not huecos_detalle.empty:
        tablas_huecos_detalle.append(huecos_detalle)

cobertura_total = pd.concat(tablas_cobertura, ignore_index=True)
estadistica_total = pd.concat(tablas_estadisticas, ignore_index=True)
huecos_resumen_total = pd.concat(tablas_huecos_resumen, ignore_index=True)
huecos_detalle_total = pd.concat(tablas_huecos_detalle, ignore_index=True) if tablas_huecos_detalle else pd.DataFrame()

print("Tabla de cobertura temporal por variable y estación:")
display(cobertura_total.style.apply(estilo_estado, axis=1))

print("Resumen estadístico por variable y estación:")
display(estadistica_total)

print("Resumen de huecos temporales por variable y estación:")
display(huecos_resumen_total.sort_values(["Variable", "Hueco_mas_largo_dias", "Total_dias_faltantes"], ascending=[True, False, False]))

cobertura_total.to_csv(CARPETA_SALIDAS / "02_cobertura_temporal_por_estacion.csv", index=False, encoding="utf-8-sig")
estadistica_total.to_csv(CARPETA_SALIDAS / "03_resumen_estadistico_por_estacion.csv", index=False, encoding="utf-8-sig")
huecos_resumen_total.to_csv(CARPETA_SALIDAS / "04_resumen_huecos_por_estacion.csv", index=False, encoding="utf-8-sig")
if not huecos_detalle_total.empty:
    huecos_detalle_total.to_csv(CARPETA_SALIDAS / "05_detalle_huecos_temporales.csv", index=False, encoding="utf-8-sig")


### Interpretación
La tabla clasifica cada estación como **Completa**, **Amarilla** o **Roja**. Esta clasificación permite identificar cuáles estaciones pueden conservarse directamente y cuáles requieren revisión antes de integrarlas o usarlas en modelos predictivos.

## 7. Gráficos comparativos por variable

In [ ]:
def graficar_variable(df_var, variable):
    display(Markdown(f"### Variable: `{variable}`"))
    cobertura = resumen_cobertura(df_var)

    # Boxplot por estación
    estaciones = sorted(df_var["Estacion"].dropna().unique())
    if len(estaciones) > 1:
        datos = [df_var.loc[df_var["Estacion"] == est, "Valor"].dropna() for est in estaciones]
        fig, ax = plt.subplots(figsize=(max(12, len(estaciones)*0.8), 5))
        ax.boxplot(datos, labels=estaciones, showfliers=True)
        ax.set_title(f"Distribución de {variable} por estación — {CIUDAD} {ANIO}")
        ax.set_xlabel("Estación")
        ax.set_ylabel(variable)
        plt.xticks(rotation=90)
        plt.tight_layout()
        plt.show()
    else:
        fig, ax = plt.subplots(figsize=(12, 5))
        df_var["Valor"].dropna().hist(bins=30, ax=ax)
        ax.set_title(f"Distribución de {variable} — {estaciones[0] if estaciones else ''}")
        ax.set_xlabel(variable)
        ax.set_ylabel("Frecuencia")
        plt.tight_layout()
        plt.show()

    # Faltantes por estación
    fig, ax = plt.subplots(figsize=(max(12, len(cobertura)*0.7), 5))
    ax.bar(cobertura["Estacion"], cobertura["Dias_faltantes"])
    ax.set_title(f"Días faltantes por estación — {variable}")
    ax.set_xlabel("Estación")
    ax.set_ylabel("Días faltantes")
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()

    # Serie temporal comparativa
    fig, ax = plt.subplots(figsize=(16, 5))
    for estacion, g in df_var.groupby("Estacion"):
        ax.plot(g["Fecha"], g["Valor"], marker=".", linewidth=1, alpha=0.75, label=estacion)
    ax.set_title(f"Serie temporal de {variable} por estación — {CIUDAD} {ANIO}")
    ax.set_xlabel("Fecha")
    ax.set_ylabel(variable)
    if df_var["Estacion"].nunique() <= 12:
        ax.legend(loc="best", fontsize=8)
    else:
        ax.legend(loc="upper left", bbox_to_anchor=(1, 1), fontsize=8)
    plt.tight_layout()
    plt.show()

    # Matriz de presencia/ausencia
    fechas = calendario_esperado()
    matriz = pd.DataFrame(index=fechas)
    for estacion in estaciones:
        obs = set(pd.DatetimeIndex(df_var[df_var["Estacion"] == estacion]["Fecha"].unique()))
        matriz[estacion] = [1 if f in obs else 0 for f in fechas]

    fig, ax = plt.subplots(figsize=(16, max(3, len(estaciones)*0.35)))
    ax.imshow(matriz.T.values, aspect="auto", interpolation="nearest")
    ax.set_title(f"Matriz de presencia/ausencia — {variable}")
    ax.set_yticks(range(len(estaciones)))
    ax.set_yticklabels(estaciones)
    ticks = np.linspace(0, len(fechas)-1, min(12, len(fechas))).astype(int)
    ax.set_xticks(ticks)
    ax.set_xticklabels([fechas[i].strftime("%Y-%m-%d") for i in ticks], rotation=45, ha="right")
    ax.set_xlabel("Fecha")
    ax.set_ylabel("Estación")
    plt.tight_layout()
    plt.show()

for variable, df_var in bases.items():
    graficar_variable(df_var, variable)


### Interpretación
Los gráficos permiten observar la dispersión de cada variable, los faltantes por estación, la evolución temporal y los cortes o huecos de medición durante el año.

## 8. Análisis detallado por estación: serie, ACF, PACF, ADF y STL

In [ ]:
ANALISIS_DETALLADO_POR_ESTACION = True
MAX_ESTACIONES_POR_VARIABLE = None  # None = todas. Puedes usar 5 si quieres una salida más corta.


def prueba_adf_segura(serie):
    s = serie.dropna()
    if len(s) < 20 or s.nunique() < 2:
        return {
            "ADF_statistic": np.nan,
            "p_value": np.nan,
            "Resultado": "No aplica: serie corta o constante"
        }
    try:
        res = adfuller(s, autolag="AIC")
        return {
            "ADF_statistic": res[0],
            "p_value": res[1],
            "Resultado": "Estacionaria" if res[1] < 0.05 else "No estacionaria"
        }
    except Exception as e:
        return {"ADF_statistic": np.nan, "p_value": np.nan, "Resultado": f"Error: {e}"}


def analizar_estacion_variable(df_var, variable, estacion):
    s = serie_diaria_estacion(df_var, estacion)
    display(Markdown(f"### {variable} — Estación: `{estacion}`"))

    resumen = pd.DataFrame([{
        "Variable": variable,
        "Estacion": estacion,
        "Fecha_min": s.dropna().index.min() if s.notna().any() else pd.NaT,
        "Fecha_max": s.dropna().index.max() if s.notna().any() else pd.NaT,
        "Dias_esperados": len(s),
        "Dias_con_dato": int(s.notna().sum()),
        "Dias_faltantes": int(s.isna().sum()),
        "%_faltantes": round(s.isna().mean()*100, 2),
        "Media": round(s.mean(), 3),
        "Mediana": round(s.median(), 3),
        "Desviacion": round(s.std(), 3),
        "Minimo": round(s.min(), 3),
        "Maximo": round(s.max(), 3),
    }])
    display(resumen)

    # Serie temporal + promedio móvil
    fig, ax = plt.subplots(figsize=(16, 5))
    ax.plot(s.index, s.values, marker=".", linewidth=1, label="Dato observado")
    ax.plot(s.rolling(7, min_periods=1).mean().index, s.rolling(7, min_periods=1).mean().values, linewidth=2, label="Promedio móvil 7 días")
    ax.set_title(f"Serie temporal diaria — {variable} — {estacion}")
    ax.set_xlabel("Fecha")
    ax.set_ylabel(variable)
    ax.legend()
    plt.tight_layout()
    plt.show()

    # Boxplot mensual
    df_mes = s.to_frame("Valor").reset_index().rename(columns={"index": "Fecha"})
    df_mes["Mes"] = df_mes["Fecha"].dt.month
    datos_mes = [df_mes.loc[df_mes["Mes"] == m, "Valor"].dropna() for m in range(1, 13)]
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.boxplot(datos_mes, labels=list(range(1, 13)), showfliers=True)
    ax.set_title(f"Distribución mensual — {variable} — {estacion}")
    ax.set_xlabel("Mes")
    ax.set_ylabel(variable)
    plt.tight_layout()
    plt.show()

    # ACF y PACF
    s_acf = s.dropna()
    if len(s_acf) > 30 and s_acf.nunique() > 2:
        lags = min(40, max(10, len(s_acf)//2 - 1))
        fig, ax = plt.subplots(figsize=(12, 4))
        plot_acf(s_acf, lags=lags, ax=ax)
        ax.set_title(f"ACF — {variable} — {estacion}")
        plt.tight_layout()
        plt.show()

        fig, ax = plt.subplots(figsize=(12, 4))
        plot_pacf(s_acf, lags=lags, ax=ax, method="ywm")
        ax.set_title(f"PACF — {variable} — {estacion}")
        plt.tight_layout()
        plt.show()
    else:
        print("No se genera ACF/PACF porque la serie tiene pocos datos válidos o es constante.")

    # ADF
    adf = prueba_adf_segura(s)
    display(pd.DataFrame([adf]))

    # STL con interpolación corta para huecos pequeños
    s_interp = s.interpolate(method="time", limit=HUECO_CORTO_MAX_DIAS, limit_direction="both")
    if s_interp.notna().sum() >= 60 and s_interp.isna().sum() == 0 and s_interp.nunique() > 2:
        try:
            stl = STL(s_interp, period=7, robust=True).fit()
            fig = stl.plot()
            fig.set_size_inches(14, 8)
            fig.suptitle(f"Descomposición STL semanal — {variable} — {estacion}", y=1.02)
            plt.tight_layout()
            plt.show()
        except Exception as e:
            print("No se pudo generar STL:", e)
    else:
        print("No se genera STL porque la serie conserva huecos largos, tiene pocos datos o es constante.")


resultados_adf = []
if ANALISIS_DETALLADO_POR_ESTACION:
    for variable, df_var in bases.items():
        estaciones = sorted(df_var["Estacion"].dropna().unique())
        if MAX_ESTACIONES_POR_VARIABLE is not None:
            estaciones = estaciones[:MAX_ESTACIONES_POR_VARIABLE]
        for estacion in estaciones:
            analizar_estacion_variable(df_var, variable, estacion)
            s = serie_diaria_estacion(df_var, estacion)
            adf = prueba_adf_segura(s)
            adf.update({"Variable": variable, "Estacion": estacion})
            resultados_adf.append(adf)

resultados_adf_df = pd.DataFrame(resultados_adf)
if not resultados_adf_df.empty:
    cols = ["Variable", "Estacion", "ADF_statistic", "p_value", "Resultado"]
    resultados_adf_df = resultados_adf_df[cols]
    display(resultados_adf_df)
    resultados_adf_df.to_csv(CARPETA_SALIDAS / "06_resultados_adf_por_estacion.csv", index=False, encoding="utf-8-sig")


### Interpretación
Esta sección reproduce el análisis por estación del EDA original. La ACF y PACF ayudan a identificar dependencia temporal; la prueba ADF orienta si la serie es estacionaria; y la STL permite separar tendencia, estacionalidad y residuo cuando la serie tiene continuidad suficiente.

## 9. Estaciones recomendadas según continuidad temporal

In [ ]:
recomendadas = cobertura_total[
    (cobertura_total["Estado"].isin(["Completa", "Amarilla"])) &
    (cobertura_total["Dias_faltantes"] <= HUECO_CORTO_MAX_DIAS)
].copy()

print(f"Estaciones con huecos de {HUECO_CORTO_MAX_DIAS} días o menos:")
display(recomendadas.sort_values(["Variable", "Dias_faltantes", "Estacion"]))
recomendadas.to_csv(CARPETA_SALIDAS / "07_estaciones_recomendadas_huecos_cortos.csv", index=False, encoding="utf-8-sig")


### Interpretación
Este filtro ayuda a decidir qué estaciones pueden pasar a una fase posterior de integración, imputación o modelado. Las estaciones con huecos cortos suelen ser más manejables porque permiten interpolación temporal limitada sin alterar fuertemente la dinámica de la serie.

## 10. Integración multivariable por estación común

In [ ]:
def construir_base_integrada(bases_dict, estaciones_objetivo=None):
    conjuntos = {var: set(df["Estacion"].dropna().unique()) for var, df in bases_dict.items()}
    estaciones_comunes = set.intersection(*conjuntos.values()) if conjuntos else set()

    if estaciones_objetivo is None:
        estaciones_objetivo = sorted(estaciones_comunes)
    else:
        estaciones_objetivo = sorted(set(estaciones_objetivo))

    print("Estaciones por variable:")
    estaciones_por_variable = pd.DataFrame([
        {"Variable": var, "Cantidad_estaciones": len(estaciones), "Estaciones": ", ".join(sorted(estaciones))}
        for var, estaciones in conjuntos.items()
    ])
    display(estaciones_por_variable)

    print("\nEstaciones comunes entre TODAS las variables:")
    display(pd.DataFrame({"Estacion_comun": sorted(estaciones_comunes)}))

    if not estaciones_objetivo:
        print("No hay estaciones comunes entre todas las variables. No se construye base integrada total.")
        return pd.DataFrame(), estaciones_por_variable, pd.DataFrame({"Estacion_comun": sorted(estaciones_comunes)})

    fechas = calendario_esperado()
    base_idx = pd.MultiIndex.from_product([estaciones_objetivo, fechas], names=["Estacion", "Fecha"]).to_frame(index=False)
    base_idx.insert(0, "Ciudad", CIUDAD)
    base_idx.insert(1, "Año", ANIO)

    integrada = base_idx.copy()
    for var, df in bases_dict.items():
        tmp = df[df["Estacion"].isin(estaciones_objetivo)][["Estacion", "Fecha", "Valor"]].copy()
        tmp = tmp.rename(columns={"Valor": var})
        integrada = integrada.merge(tmp, on=["Estacion", "Fecha"], how="left")

    return integrada, estaciones_por_variable, pd.DataFrame({"Estacion_comun": sorted(estaciones_comunes)})

base_integrada, estaciones_por_variable, estaciones_comunes_df = construir_base_integrada(bases)

print("Dimensión de la base integrada:", base_integrada.shape)
display(base_integrada.head())

if not base_integrada.empty:
    display(base_integrada.isna().sum().to_frame("Nulos_por_columna"))
    base_integrada.to_csv(CARPETA_SALIDAS / f"08_base_integrada_{CIUDAD.lower().replace(' ', '_')}_{ANIO}.csv", index=False, encoding="utf-8-sig")

estaciones_por_variable.to_csv(CARPETA_SALIDAS / "09_estaciones_por_variable.csv", index=False, encoding="utf-8-sig")
estaciones_comunes_df.to_csv(CARPETA_SALIDAS / "10_estaciones_comunes_todas_las_variables.csv", index=False, encoding="utf-8-sig")


### Interpretación
La base integrada solo se construye con estaciones presentes en todas las variables. Esto evita mezclar mediciones de estaciones distintas y permite que cada fila represente una fecha, una estación y todas las variables disponibles para análisis multivariable o modelado.

## 11. Diagnóstico de la base integrada

In [ ]:
if not base_integrada.empty:
    variables_integradas = [c for c in base_integrada.columns if c not in ["Ciudad", "Año", "Estacion", "Fecha"]]

    resumen_integrada = []
    for estacion, g in base_integrada.groupby("Estacion"):
        fila = {"Estacion": estacion, "Registros": len(g)}
        for var in variables_integradas:
            fila[f"{var}_datos_validos"] = int(g[var].notna().sum())
            fila[f"{var}_faltantes"] = int(g[var].isna().sum())
            fila[f"{var}_%faltantes"] = round(g[var].isna().mean()*100, 2)
        resumen_integrada.append(fila)

    resumen_integrada = pd.DataFrame(resumen_integrada)
    display(resumen_integrada)
    resumen_integrada.to_csv(CARPETA_SALIDAS / "11_diagnostico_base_integrada.csv", index=False, encoding="utf-8-sig")

    # Correlación entre variables integradas
    corr = base_integrada[variables_integradas].corr(numeric_only=True)
    display(corr.round(3))
    corr.to_csv(CARPETA_SALIDAS / "12_correlacion_base_integrada.csv", encoding="utf-8-sig")

    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(corr.values, vmin=-1, vmax=1)
    ax.set_xticks(range(len(corr.columns)))
    ax.set_xticklabels(corr.columns, rotation=90)
    ax.set_yticks(range(len(corr.index)))
    ax.set_yticklabels(corr.index)
    ax.set_title(f"Matriz de correlación — base integrada {CIUDAD} {ANIO}")
    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()
else:
    print("No hay base integrada para diagnosticar.")


### Interpretación
Este diagnóstico permite identificar qué variables quedan más completas en la base integrada y explorar relaciones lineales entre contaminantes y variables meteorológicas.

## 12. Conclusión automática del EDA

In [ ]:
def generar_conclusion():
    lineas = []
    lineas.append(f"Para la ciudad de {CIUDAD}, durante el año {ANIO}, se analizaron {len(bases)} variables: {', '.join(bases.keys())}.")
    lineas.append(f"La base larga consolidada contiene {len(base_larga):,} registros y {base_larga['Estacion'].nunique()} estaciones únicas considerando todas las variables.")

    completas = cobertura_total[cobertura_total["Estado"] == "Completa"]
    amarillas = cobertura_total[cobertura_total["Estado"] == "Amarilla"]
    rojas = cobertura_total[cobertura_total["Estado"] == "Roja"]
    lineas.append(f"Según el diagnóstico temporal, se identificaron {len(completas)} combinaciones variable-estación completas, {len(amarillas)} amarillas y {len(rojas)} rojas.")

    if not estaciones_comunes_df.empty and estaciones_comunes_df["Estacion_comun"].notna().any():
        comunes = estaciones_comunes_df["Estacion_comun"].dropna().tolist()
        lineas.append(f"Las estaciones comunes entre todas las variables son: {', '.join(comunes)}.")
    else:
        lineas.append("No se encontraron estaciones comunes entre todas las variables, por lo que la integración total requiere ajustar el conjunto de variables o estaciones.")

    if not huecos_resumen_total.empty:
        max_hueco = huecos_resumen_total.sort_values("Hueco_mas_largo_dias", ascending=False).head(1)
        if len(max_hueco):
            r = max_hueco.iloc[0]
            lineas.append(f"El hueco temporal más largo se presentó en {r['Variable']} - {r['Estacion']}, con {int(r['Hueco_mas_largo_dias'])} días consecutivos sin dato.")

    lineas.append("Estos resultados permiten seleccionar las estaciones con mejor continuidad para fases posteriores de imputación, integración multivariable y modelado predictivo.")
    return "\n\n".join(lineas)

conclusion = generar_conclusion()
display(Markdown(conclusion))

with open(CARPETA_SALIDAS / "13_conclusion_eda.txt", "w", encoding="utf-8") as f:
    f.write(conclusion)


## 13. Archivos generados
Al ejecutar el notebook, las tablas principales se guardan automáticamente en la carpeta de salida definida en `CARPETA_SALIDAS`. Estos archivos se pueden usar para redactar el capítulo de resultados, construir tablas en Word o alimentar la fase de modelado.

In [ ]:
print("Carpeta de salidas:", CARPETA_SALIDAS.resolve())
for archivo in sorted(CARPETA_SALIDAS.glob("*")):
    print("-", archivo.name)
